# Lab 6: API Gateway + Lambda Proxy + Frontend

Front the runtime with a REST API and a browser UI, as notebooks:
- a **Lambda proxy** that calls `invoke_agent_runtime` (with CORS),
- an **API Gateway** REST endpoint (POST /careconnect),
- a **Streamlit** frontend (mirrors the sample's lab-05 approach in SageMaker).

> API Gateway + WAF are fully boto3-scriptable but verbose; the cells below create a
> minimal working POST endpoint. Add WAF managed rules + rate limiting before any real use.

### Step 1: Create the proxy Lambda

In [ ]:
import boto3, io, zipfile, json
import lab_helpers.utils as u
lambda_client = boto3.client("lambda", region_name=u.REGION)
account = u.get_aws_account_id()
runtime_arn = u.get_ssm_parameter(f"{u.SSM_PREFIX}/runtime_arn")

PROXY_SRC = f'''
import json, re, boto3
AGENT_RUNTIME_ARN = "{runtime_arn}"
AWS_REGION = "{u.REGION}"
CORS = {{"Content-Type":"application/json","Access-Control-Allow-Origin":"*",
        "Access-Control-Allow-Headers":"Content-Type","Access-Control-Allow-Methods":"OPTIONS,POST"}}
def _reply(s,o): return {{"statusCode":s,"headers":CORS,"body":json.dumps(o)}}
def _sse(raw):
    out=[]
    for ln in raw.splitlines():
        ln=ln.strip()
        if ln.startswith("data:"):
            try: evt=json.loads(ln[5:].strip())
            except ValueError: continue
            t=evt.get("event",{{}}).get("contentBlockDelta",{{}}).get("delta",{{}}).get("text")
            if t: out.append(t)
    return "".join(out)
def lambda_handler(event, context):
    if event.get("httpMethod")=="OPTIONS": return {{"statusCode":200,"headers":CORS,"body":""}}
    body=event.get("body") or "{{}}"
    body=json.loads(body) if isinstance(body,str) else body
    prompt=(body.get("prompt") or "").strip()
    if not prompt: return _reply(400,{{"error":"Missing prompt"}})
    c=boto3.client("bedrock-agentcore",region_name=AWS_REGION)
    r=c.invoke_agent_runtime(agentRuntimeArn=AGENT_RUNTIME_ARN,
        payload=json.dumps({{"prompt":prompt}}).encode())
    raw=r["response"].read()
    raw=raw.decode() if isinstance(raw,(bytes,bytearray)) else str(raw)
    ans=re.sub(r"<thinking\\b[^>]*>.*?</thinking>","",_sse(raw) or raw,flags=re.DOTALL).strip()
    return _reply(200,{{"answer":ans}})
'''

buf=io.BytesIO()
with zipfile.ZipFile(buf,"w") as z: z.writestr("lambda_function.py", PROXY_SRC)
buf.seek(0)

proxy_role = u._create_role(
    u.name("CareConnectProxyRole"), "lambda.amazonaws.com",
    {"Version":"2012-10-17","Statement":[
        {"Effect":"Allow","Action":["logs:CreateLogGroup","logs:CreateLogStream","logs:PutLogEvents"],"Resource":"*"},
        {"Effect":"Allow","Action":["bedrock-agentcore:InvokeAgentRuntime"],"Resource":"*"}]},
    u.name("CareConnectProxyPolicy"))

try:
    fn=lambda_client.create_function(FunctionName=u.PROXY_LAMBDA, Runtime="python3.12",
        Role=proxy_role, Handler="lambda_function.lambda_handler",
        Code={"ZipFile":buf.read()}, Timeout=300)
    print("Created proxy:", fn["FunctionArn"])
except lambda_client.exceptions.ResourceConflictException:
    fn=lambda_client.get_function(FunctionName=u.PROXY_LAMBDA)["Configuration"]
    print("Reusing proxy:", fn["FunctionArn"])
proxy_arn=fn["FunctionArn"]

### Step 2: Create the REST API (POST /careconnect) and wire the proxy

In [ ]:
apigw = boto3.client("apigateway", region_name=u.REGION)
api = apigw.create_rest_api(name=u.name("careconnect-api"),
                            endpointConfiguration={"types":["REGIONAL"]})
api_id = api["id"]
root = apigw.get_resources(restApiId=api_id)["items"][0]["id"]
res = apigw.create_resource(restApiId=api_id, parentId=root, pathPart="careconnect")["id"]
apigw.put_method(restApiId=api_id, resourceId=res, httpMethod="POST", authorizationType="NONE")
uri = f"arn:aws:apigateway:{u.REGION}:lambda:path/2015-03-31/functions/{proxy_arn}/invocations"
apigw.put_integration(restApiId=api_id, resourceId=res, httpMethod="POST",
                      type="AWS_PROXY", integrationHttpMethod="POST", uri=uri)
apigw.create_deployment(restApiId=api_id, stageName="prod")

account = u.get_aws_account_id()
lambda_client.add_permission(FunctionName=u.PROXY_LAMBDA,
    StatementId="apigw-invoke", Action="lambda:InvokeFunction",
    Principal="apigateway.amazonaws.com",
    SourceArn=f"arn:aws:execute-api:{u.REGION}:{account}:{api_id}/*/POST/careconnect")

invoke_url=f"https://{api_id}.execute-api.{u.REGION}.amazonaws.com/prod/careconnect"
u.put_ssm_parameter(f"{u.SSM_PREFIX}/api_url", invoke_url)
print("API URL:", invoke_url)

### Step 3: Test the endpoint

In [ ]:
import urllib.request, json
req=urllib.request.Request(invoke_url,
    data=json.dumps({"prompt":"How should I prepare for my colonoscopy?"}).encode(),
    headers={"Content-Type":"application/json"}, method="POST")
print(urllib.request.urlopen(req).read().decode())

### Step 4: Streamlit frontend (SageMaker)

The sample serves a Streamlit chat app from the notebook. A ready-to-run app is provided in
`lab_helpers/frontend/app.py`. It reads the API URL from SSM. Launch it from a terminal:

```bash
export CARECONNECT_API_URL=$(aws ssm get-parameter --name /app/careconnect/agentcore/api_url --query Parameter.Value --output text)
streamlit run lab_helpers/frontend/app.py --server.port 8501
```

In SageMaker, open the app via the Jupyter proxy URL
(`https://<studio-domain>/jupyter/default/proxy/8501/`).

## Lab 6 complete ✅

REST endpoint + proxy + Streamlit UI, all `-sdk`. Add WAF/rate-limiting before real use.